# ML Baseline - Delay Prediction
This notebook walks through the data preparation, chronological split, preprocessing, and model training for the logistic regression baseline used in production.

In [1]:
import sys

sys.path.append("..")

import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from ml.dataset import get_training_data
from ml.pipeline import get_preprocessing_pipeline, split_data
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

ModuleNotFoundError: No module named 'duckdb'

## Load Data
We load the pre-calculated features and target from DuckDB. We have explicitly removed all leakage features (like actual arrival/departure).

In [ ]:
conn = duckdb.connect("../data/curated/supply_chain.db", read_only=True)
df = get_training_data(conn)
conn.close()
df.head()

## Target Class Distribution
Let's check the class balance for our target variable.

In [ ]:
print(df["target"].value_counts(normalize=True))
sns.countplot(data=df, x="target")
plt.title("Target Class Balance")
plt.show()

## Split Data Chronologically
We use a 70% Train, 15% Validation, 15% Test chronological split.

In [ ]:
train_df, val_df, test_df = split_data(df)
print(f"Train size: {len(train_df)}")
print(f"Validation size: {len(val_df)}")
print(f"Test size: {len(test_df)}")

## Train Baseline Model
We apply our scikit-learn preprocessing pipeline (handling NaNs and One-Hot Encoding) and fit the Logistic Regression baseline.

In [ ]:
X_train = train_df.drop(columns=["target"])
y_train = train_df["target"]

pipeline = get_preprocessing_pipeline()
pipeline.fit(X_train, y_train)

## Evaluate on Test Set
Finally, we evaluate on the untouched test set.

In [ ]:
X_test = test_df.drop(columns=["target"])
y_test = test_df["target"]

y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.5).astype(int)

print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()